In [ ]:
import subprocess
import re
import csv
import pandas as pd
import plotly.graph_objects as go

def run_command(block_size, num_blocks):
    """Run the command with given block_size and num_blocks, and return runtime (in ms) from 'Mining Completed'."""
    command = [
        "./build/cuDPEFIM",
        "--input", "accidents_utility_spmf.txt",
        "--minutil", "15000000",
        "--threads", "8",
        "--managed", "true",
        "--block_size", str(block_size),
        "--num_blocks", str(num_blocks)
    ]
    
    try:
        result = subprocess.run(command, capture_output=True, text=True, check=True)
        output = result.stdout
        
        runtime = None
        # Look for the line containing "Mining Completed"
        for line in output.splitlines():
            if "Mining Completed" in line:
                match = re.search(r"Mining Completed\s+(\d+)\s+(\d+)", line)
                if match:
                    runtime = int(match.group(1))
                    break
        if runtime is None:
            print(f"[Warning] Runtime not found for block_size={block_size}, num_blocks={num_blocks}")
        return runtime
    except subprocess.CalledProcessError as e:
        print(f"Command failed for block_size={block_size}, num_blocks={num_blocks}: {e}")
        return None

def generate_values(start, end, doubling=True, include_end=False):
    """Generate values starting at 'start' doubling until exceeding 'end'. Optionally append 'end' if not reached."""
    values = []
    v = start
    while v <= end:
        values.append(v)
        v *= 2
    if include_end and values[-1] != end:
        values.append(end)
    return values

def run_experiments():
    # Generate values for block_size: 1,2,4,...,512.
    block_sizes = generate_values(128, 512, doubling=True, include_end=False)
    # Generate values for num_blocks: 1,2,4,..., and include 43 if not reached.
    num_blocks_list = generate_values(16, 43, doubling=True, include_end=True)
    
    results = []
    total_runs = len(block_sizes) * len(num_blocks_list)
    run_count = 1
    
    for bs in block_sizes:
        for nb in num_blocks_list:
            print(f"Run {run_count} of {total_runs}: block_size={bs}, num_blocks={nb}")
            runtime = run_command(bs, nb)
            if runtime is not None:
                results.append({"block_size": bs, "num_blocks": nb, "runtime": runtime})
            run_count += 1
    
    # Save results to CSV.
    csv_filename = "results.csv"
    with open(csv_filename, "w", newline="") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=["block_size", "num_blocks", "runtime"])
        writer.writeheader()
        writer.writerows(results)
    print(f"Results saved to {csv_filename}")

def plot_results():
    # Read the results from the CSV file.
    csv_filename = "results.csv"
    df = pd.read_csv(csv_filename)
    
    # Create a 3D plot and connect the dots for each fixed num_blocks.
    fig = go.Figure()
    
    for nb in sorted(df["num_blocks"].unique()):
        df_subset = df[df["num_blocks"] == nb].sort_values("block_size")
        fig.add_trace(go.Scatter3d(
            x = df_subset["block_size"],
            y = df_subset["num_blocks"],
            z = df_subset["runtime"],
            mode = "lines+markers",
            name = f"Num Blocks: {nb}"
        ))
    
    fig.update_layout(
        title="Runtime (ms) vs Block Size and Num Blocks",
        scene = dict(
            xaxis_title="Block Size",
            yaxis_title="Num Blocks",
            zaxis_title="Runtime (ms)"
        )
    )
    
    # Save the interactive plot as an HTML file.
    html_filename = "3d_runtime_graph.html"
    fig.write_html(html_filename)
    print(f"Interactive 3D plot saved to {html_filename}")
    
    fig.show()

def main():
    # Run the experiments and save results to CSV.
    # run_experiments()
    # Read the results from the CSV file and create the interactive plot.
    plot_results()

if __name__ == "__main__":
    main()


Run 1 of 9: block_size=128, num_blocks=16
Run 2 of 9: block_size=128, num_blocks=32
Run 3 of 9: block_size=128, num_blocks=43
Run 4 of 9: block_size=256, num_blocks=16
Run 5 of 9: block_size=256, num_blocks=32
Run 6 of 9: block_size=256, num_blocks=43
Run 7 of 9: block_size=512, num_blocks=16
Run 8 of 9: block_size=512, num_blocks=32
Run 9 of 9: block_size=512, num_blocks=43
Results saved to results.csv
Interactive 3D plot saved to 3d_runtime_graph.html


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed